In [2]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet
from torchvision.utils import save_image
from torch.utils.data import Subset
from matplotlib import pyplot as plt

# -----------------------------
# Utilities
# -----------------------------
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    #torch.cuda.manual_seed_all(seed)

# -----------------------------
# U-Net model
# -----------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=1, base_ch=64):
        super().__init__()
        self.down1 = DoubleConv(in_ch, base_ch)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(base_ch, base_ch*2)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(base_ch*2, base_ch*4)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(base_ch*4, base_ch*8)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_ch*8, base_ch*16)

        self.up4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_ch*16, base_ch*8)

        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_ch*8, base_ch*4)

        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_ch*4, base_ch*2)

        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_ch*2, base_ch)

        self.head = nn.Conv2d(base_ch, n_classes, kernel_size=1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool1(c1)

        c2 = self.down2(p1)
        p2 = self.pool2(c2)

        c3 = self.down3(p2)
        p3 = self.pool3(c3)

        c4 = self.down4(p3)
        p4 = self.pool4(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        d4 = self.dec4(torch.cat([u4, c4], dim=1))

        u3 = self.up3(d4)
        d3 = self.dec3(torch.cat([u3, c3], dim=1))

        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, c2], dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, c1], dim=1))

        logits = self.head(d1)
        return logits

# -----------------------------
# Dataset wrapper for binary masks
# -----------------------------
class OxfordPetSegBinary(Dataset):
    """
    Wrap OxfordIIITPet(split='trainval'/'test', target types='segmentation')
    Convert trimap {1,2,3} -> binary mask {0,1} with (mask>1)->1
    """
    def __init__(self, root, split="trainval", image_size=256, download=True):
        self.ds = OxfordIIITPet(
            root=root, split=split, target_types="segmentation", download=download
        )
        self.ds = Subset(self.ds, range(300))
        self.img_transform = T.Compose([
            T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BILINEAR),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])
        self.mask_transform = T.Resize((image_size, image_size), interpolation=T.InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, mask = self.ds[idx]  # PIL.Image, PIL.Image (mode='L', values in {1,2,3})
        img = self.img_transform(img)
        mask = self.mask_transform(mask)
        mask_np = np.array(mask, dtype=np.uint8)  # still {1,2,3}
        bin_mask = (mask_np > 1).astype(np.float32)  # {0,1}
        bin_mask = torch.from_numpy(bin_mask).unsqueeze(0)  # (1,H,W)
        return img, bin_mask

# -----------------------------
# Loss / Metrics
# -----------------------------
class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        # Dice (soft)
        num = 2.0 * (probs * targets).sum(dim=(2,3)) + self.smooth
        den = (probs + targets).sum(dim=(2,3)) + self.smooth
        dice = (num / den).mean()
        return bce + (1.0 - dice)

@torch.no_grad()
def dice_iou(logits, targets, thr=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > thr).float()
    inter = (preds * targets).sum(dim=(2,3))
    union = (preds + targets - preds*targets).sum(dim=(2,3))
    dice = (2*inter / (preds.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + 1e-7)).mean().item()
    iou = (inter / (union + 1e-7)).mean().item()
    return dice, iou

# -----------------------------
# Training / Validation
# -----------------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for imgs, masks in tqdm(loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, total_iou, n = 0.0, 0.0, 0.0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = criterion(logits, masks)
        dice, iou = dice_iou(logits, masks)
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_dice += dice * bs
        total_iou  += iou  * bs
        n += bs
    return total_loss / n, total_dice / n, total_iou / n

# -----------------------------
# Inference & saving masks
# -----------------------------
@torch.no_grad()
def run_inference(model, loader, device, out_dir, num_batches=3, thr=0.5):
    model.eval()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    for bidx, (imgs, masks) in enumerate(loader):
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        preds = (probs > thr).float()

        for i in range(imgs.size(0)):
            mask = masks[i].to(device)
            img = ((imgs[i] * torch.tensor([0.229,0.224,0.225], device=device).view(3,1,1)) +
                   torch.tensor([0.485,0.456,0.406], device=device).view(3,1,1)).clamp(0,1)
            grid = torch.cat([
                img,
                mask.repeat(3,1,1),
                preds[i].repeat(3,1,1),
            ], dim=2)
            save_image(grid.cpu(), out_dir / f"sample_{bidx:02d}_{i:02d}.png")
            saved += 1
        if bidx + 1 >= num_batches:
            break
    return saved

# -----------------------------
# Main (train / infer)
# -----------------------------
def main():
    val_ratio = 0.1  # validation split ratio
    batch_size = 64  # tuned for MPS memory constraints
    infer_only = False
    weights = "./runs_unet_pet/best.pt"
    lr = 1e-3
    epochs = 20
    set_seed(42)

    device = get_device()
    print(f"Using device: {device}")
    save_dir = Path("./runs_unet_pet")
    save_dir.mkdir(parents=True, exist_ok=True)

    # Dataset
    full_ds = OxfordPetSegBinary(root='./pet_seg_data', split="trainval", image_size=256, download=True)
    print(f"Dataset size: {len(full_ds)} images")
    val_len = max(1, int(len(full_ds) * val_ratio))
    train_len = len(full_ds) - val_len
    train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # Model
    model = UNet(in_ch=3, n_classes=1, base_ch=64).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = BCEDiceLoss()
    best_iou = 0.0

    if infer_only:
        assert weights and os.path.exists(weights), "請提供有效的權重檔路徑"
        ckpt = torch.load(weights, map_location=device)
        model.load_state_dict(ckpt["model"])
        saved = run_inference(model, val_loader, device, out_dir=save_dir / "preds", num_batches=3)
        print(f"Saved {saved} prediction samples to {save_dir/'preds'}")
        return

    # Training loop
    for epoch in range(1, epochs+1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_dice, va_iou = validate(model, val_loader, criterion, device)
        print(f"[Epoch {epoch:02d}/{epochs}] train_loss={tr_loss:.4f} val_loss={va_loss:.4f} dice={va_dice:.4f} iou={va_iou:.4f}")
        if va_iou > best_iou:
            best_iou = va_iou
            ckpt = {
                "model": model.state_dict(),
                "epoch": epoch,
                "val_iou": va_iou,
            }
            torch.save(ckpt, save_dir / "best.pt")

    print(f"Training done. Best IoU = {best_iou:.4f}. Weights saved to {save_dir/'best.pt'}")

    ckpt = torch.load(save_dir / "best.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    saved = run_inference(model, val_loader, device, out_dir=save_dir / "preds", num_batches=3)
    print(f"Saved {saved} prediction samples to {save_dir/'preds'}")

if __name__ == "__main__":
    main()
# --- 最後一個儲存格：訓練 60 epochs & 繪圖 & 推論 5 張 test 圖 ---
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

epochs     = 60  # 介於 50~100，滿足作業要求
lr         = 1e-3
batch_size = 32
val_ratio  = 0.1

set_seed(42)
device = get_device()

# 建立 train/val 資料集
full_ds = OxfordPetSegBinary(root='./pet_seg_data', split='trainval', image_size=256, download=False)
train_len = int(len(full_ds) * (1 - val_ratio))
val_len   = len(full_ds) - train_len
train_ds, val_ds = random_split(full_ds, [train_len, val_len],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

# 模型、優化器、損失函數
model     = UNet(in_ch=3, n_classes=1, base_ch=64).to(device)
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
criterion = BCEDiceLoss()

# 訓練與驗證迴圈
train_losses, val_losses = [], []
for epoch in range(1, epochs+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, _, _ = validate(model, val_loader, criterion, device)
    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    print(f'Epoch {epoch}/{epochs} train_loss={tr_loss:.4f} val_loss={va_loss:.4f}')

# 繪製 train/validation loss 折線圖
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs+1), train_losses, label='Train')
plt.plot(range(1, epochs+1), val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
loss_plot_path = Path('runs_unet_pet/train_val_loss.png')
loss_plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(loss_plot_path, dpi=150)
plt.show()
print(f'Saved loss plot to {loss_plot_path.resolve()}')

# 推論 5 張 test 圖並顯示
test_ds     = OxfordPetSegBinary(root='./pet_seg_data', split='test', image_size=256, download=False)
test_loader = DataLoader(test_ds, batch_size=5, shuffle=True)

pred_dir = Path('./runs_unet_pet/test_preds')
pred_dir.mkdir(parents=True, exist_ok=True)
for old_png in pred_dir.glob('*.png'):
    old_png.unlink()

_ = run_inference(model, test_loader, device,
                  out_dir=pred_dir, num_batches=1)

pred_files = sorted(pred_dir.glob('*.png'))[:5]
fig, axes = plt.subplots(1, len(pred_files), figsize=(15,5))
for ax, png_path in zip(axes, pred_files):
    ax.imshow(mpimg.imread(png_path))
    ax.set_title(png_path.name)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Saved {len(pred_files)} inference samples to {pred_dir.resolve()}')

Using device: mps
Dataset size: 300 images
Dataset size: 300 images


RuntimeError: MPS backend out of memory (MPS allocated: 17.05 GiB, other allocations: 1.00 GiB, max allowed: 18.13 GiB). Tried to allocate 512.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).